In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_ratings, load_items, load_users, GENRE_COLS

In [ ]:
ratings = load_ratings('../data/raw')
items = load_items('../data/raw')
users = load_users('../data/raw')
print(ratings.shape, items.shape, users.shape)

## Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.set_title('Rating Distribution')
plt.tight_layout()
plt.show()

print(ratings['rating'].describe())

## Sparsity

In [ ]:
n_users = ratings['user_id'].nunique()
n_items = ratings['item_id'].nunique()
n_ratings = len(ratings)
sparsity = n_ratings / (n_users * n_items)
print(f'Users: {n_users}, Items: {n_items}')
print(f'Ratings: {n_ratings}')
print(f'Density: {sparsity:.4f} ({sparsity*100:.2f}%)')

## User activity (long tail)

In [ ]:
user_counts = ratings.groupby('user_id').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_counts, bins=50, color='steelblue')
axes[0].set_xlabel('Number of ratings')
axes[0].set_ylabel('Number of users')
axes[0].set_title('User Activity Distribution')

sorted_counts = user_counts.sort_values(ascending=False).values
axes[1].plot(range(len(sorted_counts)), sorted_counts)
axes[1].set_xlabel('User rank')
axes[1].set_ylabel('# ratings (log)')
axes[1].set_yscale('log')
axes[1].set_title('User Activity - Long Tail')
plt.tight_layout()
plt.show()

print(f'min: {user_counts.min()}, max: {user_counts.max()}, median: {user_counts.median()}')

## Item popularity

In [ ]:
item_counts = ratings.groupby('item_id').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(item_counts, bins=50, color='coral')
axes[0].set_xlabel('Number of ratings')
axes[0].set_ylabel('Number of items')
axes[0].set_title('Item Popularity Distribution')

sorted_ic = item_counts.sort_values(ascending=False).values
axes[1].plot(range(len(sorted_ic)), sorted_ic)
axes[1].set_xlabel('Item rank')
axes[1].set_ylabel('# ratings (log)')
axes[1].set_yscale('log')
axes[1].set_title('Item Popularity - Long Tail')
plt.tight_layout()
plt.show()

## Avg rating by genre

In [ ]:
# merge ratings with item genres
merged = ratings.merge(items[['item_id'] + GENRE_COLS], on='item_id')

genre_avg = {}
for g in GENRE_COLS:
    mask = merged[g] == 1
    if mask.sum() > 0:
        genre_avg[g] = merged.loc[mask, 'rating'].mean()

genre_df = pd.Series(genre_avg).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
genre_df.plot(kind='bar', ax=ax, color='teal')
ax.set_ylabel('Avg Rating')
ax.set_title('Average Rating by Genre')
ax.axhline(y=ratings['rating'].mean(), color='red', linestyle='--', label='overall mean')
ax.legend()
plt.tight_layout()
plt.show()

## User demographics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(users['age'], bins=30, color='mediumpurple')
axes[0].set_xlabel('Age')
axes[0].set_title('User Age Distribution')

users['gender'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Gender Distribution')
plt.tight_layout()
plt.show()